# ĐỒ ÁN TỐT NGHIỆP: NGHIÊN CỨU XÂY DỰNG MÔ HÌNH MACHINE LEARNING TRONG ĐÁNH GIÁ VÀ DỰ BÁO RỦI RO TÍN DỤNG KHÁCH HÀNG
Giai đoạn 2: Preprocessing, Train/Test Split & Kiểm soát Data Leakage (Tuần 4)**Dataset**: UCI Credit Card Default Dataset (`UCI_Credit_Card.csv`)  **Thời gian timeline**: Tuần 4 (14/09/2026 - 20/09/2026)  
### Mục tiêu chính:
1. Thực hiện làm sạch dữ liệu sơ bộ dựa trên phát hiện của EDA (loại bỏ 35 dòng trùng lặp đặc trưng, giữ nguyên ID trong raw data).
2. Quy chuẩn các mã danh mục chưa mô tả rõ (`EDUCATION` 0, 5, 6 và `MARRIAGE` 0).
3. Thực hiện **Train/Test Split (80/20, stratify=y, random_state=42)** trước khi tính toán bất kỳ tham số chuẩn hóa nào.
4. Xây dựng **Preprocessing Pipeline (`ColumnTransformer`)** sử dụng `RobustScaler` fit **CHỈ TRÊN TẬP TRAIN** và transform tập TEST để chống **Data Leakage** tuyệt đối.
5. Nghiên cứu và chuẩn bị phương án xử lý Mất cân bằng lớp (`class_weight='balanced'`, `SMOTE` cho tập TRAIN trong các bước ML tiếp theo).
6. **Lưu ý tuân thủ**: Tuyệt đối **KHÔNG** train mô hình Machine Learning, **KHÔNG** làm Feature Engineering (Tuần 5), **KHÔNG** tuning hay tạo SHAP/Streamlit.

In [19]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, StandardScaler

# Add src module path
sys.path.append(os.path.join('..', 'src'))
from preprocessing import (
    load_raw_data, clean_duplicates, map_categorical_anomalies,
    separate_features_target, split_train_test, build_preprocessing_pipeline,
    fit_transform_data, save_processed_data
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Đã tải xong các module tiền xử lý.")

Đã tải xong các module tiền xử lý.


---## 2. Load Dữ liệu Đầu vào (Data Loading)Tải tập dữ liệu gốc từ `data/raw/UCI_Credit_Card.csv` hoặc `data/UCI_Credit_Card.csv`.

In [20]:
csv_path = os.path.join('..', 'data', 'raw', 'UCI_Credit_Card.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.join('data', 'raw', 'UCI_Credit_Card.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.join('data', 'UCI_Credit_Card.csv')

df_raw = pd.read_csv(csv_path)
print(f"Kích thước tập dữ liệu thô (Raw Data): {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")

Kích thước tập dữ liệu thô (Raw Data): 30,000 dòng x 25 cột


---## 3. Kiểm tra Dữ liệu Đầu vào (Input Data Audit)Xác nhận các thông số toàn vẹn từ bước EDA:- Tổng số quan sát: 30,000 dòng, 25 cột.- Ô dữ liệu khuyết (Missing Values): 0 ô (0.00%).- Kiểu dữ liệu: 25/25 cột là `int64`.

In [21]:
print(f"Tổng số missing values: {df_raw.isnull().sum().sum()}")
print("Kiểm tra 5 dòng đầu tiên:")
display(df_raw.head())

Tổng số missing values: 0
Kiểm tra 5 dòng đầu tiên:


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


---## 4. Xử lý Dòng Trùng lặp (Duplicate Removal)Giai đoạn EDA phát hiện **35 dòng dữ liệu trùng lặp đặc trưng** khi loại bỏ cột `ID`.Trước khi tiến hành loại bỏ, ta kiểm tra mức độ nhất quán của biến mục tiêu `default.payment.next.month` giữa các nhóm dòng trùng lặp này.

In [22]:
feature_cols = [c for c in df_raw.columns if c != 'ID']
dup_mask = df_raw.duplicated(subset=feature_cols, keep=False)
dup_df = df_raw[dup_mask].sort_values(by=feature_cols)

print(f"Tổng số dòng tham gia vào các nhóm trùng lặp: {len(dup_df)}")
print("Phân bố Target trong các dòng trùng lặp:")
print(dup_df['default.payment.next.month'].value_counts())

# Kiểm tra sự nhất quán Target
grouped = dup_df.groupby(feature_cols)['default.payment.next.month'].nunique()
inconsistent_count = (grouped > 1).sum()
print(f"Số lượng nhóm trùng lặp có Target BẤT NHẤT: {inconsistent_count}")

# Tiến hành loại bỏ 35 dòng trùng lặp (giữ lại 1 bản ghi đại diện đầu tiên)
df_cleaned, num_dups = clean_duplicates(df_raw)
print(f"\n--- THỐNG KÊ XỬ LÝ DUPLICATE ---")
print(f"Số dòng trước xử lý : {len(df_raw):,}")
print(f"Số trùng lặp loại bỏ: {num_dups}")
print(f"Số dòng sau xử lý    : {len(df_cleaned):,}")

Tổng số dòng tham gia vào các nhóm trùng lặp: 70
Phân bố Target trong các dòng trùng lặp:
default.payment.next.month
0    58
1    12
Name: count, dtype: int64
Số lượng nhóm trùng lặp có Target BẤT NHẤT: 0

--- THỐNG KÊ XỬ LÝ DUPLICATE ---
Số dòng trước xử lý : 30,000
Số trùng lặp loại bỏ: 35
Số dòng sau xử lý    : 29,965


### Nhận xét Phần 4:- 100% các nhóm trùng lặp đều có **Target hoàn toàn nhất quán** (0 nhóm bất nhất).- Việc loại bỏ 35 dòng trùng lặp giúp làm sạch tập dữ liệu, ngăn ngừa tình trạng lặp lại mẫu thông tin trùng lặp gây thiên vị (sample bias) trong quá trình huấn luyện mô hình.- Bản dữ liệu thô `UCI_Credit_Card.csv` ban đầu được giữ nguyên trong thư mục `data/raw/`.

---## 5. Xử lý Cột Định danh ID (ID Handling)- Cột `ID` là mã thứ tự định danh quản lý tài khoản khách hàng, không chứa ý nghĩa nghiệp vụ tài chính hay thuộc tính hành vi.- Nếu đưa `ID` vào không gian thuộc tính $X$, các mô hình ML (đặc biệt là cây quyết định hoặc mạng nơ-ron) có thể học vẹt chỉ số index (artifact memorization).- **Quyết định**: Loại bỏ cột `ID` khỏi tập thuộc tính $X$, nhưng **giữ nguyên cột ID trong raw dataset** để đảm bảo khả năng truy vết (traceability).

In [23]:
print("Cột ID sẽ được loại bỏ khỏi tập thuộc tính X trước khi chia Train/Test Split.")

Cột ID sẽ được loại bỏ khỏi tập thuộc tính X trước khi chia Train/Test Split.


---## 6. Xử lý Mã Danh mục Bất thường (Categorical Anomalies Handling)Phát hiện từ EDA:1. **`EDUCATION`**: Mã `0` (14 mẫu), `5` (280 mẫu), `6` (51 mẫu) chưa quy ước trong mô tả gốc của UCI (1=Grad School, 2=University, 3=High School, 4=Others).   - Tỷ lệ Target: Mã 0 (0% default), Mã 5 (6.43% default), Mã 6 (15.69% default).   - *Quyết định*: Gom toàn bộ các mã `0, 5, 6` vào mã `4 (Others)` đại diện cho nhóm học vấn khác/không xác định.2. **`MARRIAGE`**: Mã `0` (54 mẫu ~ 0.18%) chưa quy ước trong mô tả gốc (1=Married, 2=Single, 3=Others).   - *Quyết định*: Gom mã `0` vào mã `3 (Others)`.

In [24]:
print("=== THỐNG KÊ TRƯỚC KHI MAPPING ===")
print("EDUCATION:", df_cleaned['EDUCATION'].value_counts().sort_index().to_dict())
print("MARRIAGE :", df_cleaned['MARRIAGE'].value_counts().sort_index().to_dict())

df_mapped = map_categorical_anomalies(df_cleaned)

print("\n=== THỐNG KÊ SAU KHI MAPPING ===")
print("EDUCATION:", df_mapped['EDUCATION'].value_counts().sort_index().to_dict())
print("MARRIAGE :", df_mapped['MARRIAGE'].value_counts().sort_index().to_dict())

=== THỐNG KÊ TRƯỚC KHI MAPPING ===
EDUCATION: {0: 14, 1: 10563, 2: 14019, 3: 4915, 4: 123, 5: 280, 6: 51}
MARRIAGE : {0: 54, 1: 13643, 2: 15945, 3: 323}

=== THỐNG KÊ SAU KHI MAPPING ===
EDUCATION: {1: 10563, 2: 14019, 3: 4915, 4: 468}
MARRIAGE : {1: 13643, 2: 15945, 3: 377}


---## 7. Xác định Tập Thuộc tính X và Biến Mục tiêu y

In [25]:
X, y = separate_features_target(df_mapped, target_col='default.payment.next.month', id_col='ID')
print(f"Kích thước tập thuộc tính X: {X.shape[0]:,} dòng x {X.shape[1]} cột")
print(f"Kích thước biến mục tiêu y : {y.shape[0]:,} dòng")
print("Danh sách 23 thuộc tính đầu vào:", list(X.columns))

Kích thước tập thuộc tính X: 29,965 dòng x 23 cột
Kích thước biến mục tiêu y : 29,965 dòng
Danh sách 23 thuộc tính đầu vào: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


---## 8. Chia Tập Dữ liệu Train / Test (Train/Test Split)> [!IMPORTANT]> **QUY TẮC PHÒNG CHỐNG DATA LEAKAGE**:> Việc phân chia tập TRAIN và TEST được thực hiện **NGAY TẠI BƯỚC NÀY**, trước khi tính toán bất kỳ giá trị thống kê nào (Mean, Median, IQR, Min, Max).> - Tỷ lệ chia: **80% TRAIN** và **20% TEST**.> - Phân tầng: `stratify=y` để bảo toàn tỷ lệ vỡ nợ 22.13% ở cả 2 tập.> - Cố định `random_state = 42` để đảm bảo khả năng tái lập 100%.

In [26]:
X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.20, random_state=42)

print(f"Số lượng mẫu tập TRAIN: {len(X_train):,} mẫu (80.00%)")
print(f"Số lượng mẫu tập TEST : {len(X_test):,} mẫu (20.00%)")

Số lượng mẫu tập TRAIN: 23,972 mẫu (80.00%)
Số lượng mẫu tập TEST : 5,993 mẫu (20.00%)


---## 9. Kiểm tra Phân bố Target sau Split (Stratification Audit)

In [27]:
train_counts = y_train.value_counts()
train_pcts = y_train.value_counts(normalize=True) * 100

test_counts = y_test.value_counts()
test_pcts = y_test.value_counts(normalize=True) * 100

split_audit = pd.DataFrame({
    'TRAIN Count': train_counts,
    'TRAIN (%)': train_pcts.round(2),
    'TEST Count': test_counts,
    'TEST (%)': test_pcts.round(2)
})
split_audit.index = ['Không vỡ nợ (0)', 'Vỡ nợ (1)']
display(split_audit)

,TRAIN Count,TRAIN (%),TEST Count,TEST (%)
Không vỡ nợ (0),18668,77.87,4667,77.87
Vỡ nợ (1),5304,22.13,1326,22.13


### Nhận xét Phần 9:- Tỷ lệ vỡ nợ ở tập TRAIN là **22.13%** (5,304 vỡ nợ / 18,668 bình thường).- Tỷ lệ vỡ nợ ở tập TEST là **22.12%** (1,326 vỡ nợ / 4,667 bình thường).- Phân tầng `stratify=y` hoạt động chính xác tuyệt đối, đảm bảo tính đại diện đồng đều của tập TEST.

---## 10. Phân chia Nhóm Thuộc tính Số và Danh mục (Feature Categorization)- **Numerical Features (14 biến)**: `LIMIT_BAL`, `AGE`, `BILL_AMT1..6`, `PAY_AMT1..6`.- **Categorical / Ordinal Features (9 biến)**: `SEX`, `EDUCATION`, `MARRIAGE`, `PAY_0..6`.  - *Lưu ý về PAY_0..6*: Các biến này biểu thị số tháng trậm trả (-2: không dùng nợ, -1: đúng hạn, 0: trả xoay vòng, 1..8: trậm trả 1..8 tháng). Giữ nguyên giá trị số để bảo toàn thứ tự nghiêm trọng.

In [28]:
num_cols = ['LIMIT_BAL', 'AGE'] + [f'BILL_AMT{i}' for i in range(1, 7)] + [f'PAY_AMT{i}' for i in range(1, 7)]
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

print(f"Số lượng thuộc tính số (Numerical): {len(num_cols)}")
print(f"Số lượng thuộc tính danh mục (Categorical/Ordinal): {len(cat_cols)}")

Số lượng thuộc tính số (Numerical): 14
Số lượng thuộc tính danh mục (Categorical/Ordinal): 9


---## 11. Xây dựng Preprocessing Pipeline (`ColumnTransformer`)Lựa chọn Scaler:- `PAY_AMT1..6` có Skewness rất lớn ($10.6 - 30.5$) và chứa ngoại biệt cực đại (hàng trăm nghìn NTD).- `RobustScaler` sử dụng Trung vị (Median) và Khoảng tứ phân vị IQR ($Q75 - Q25$) thay vì Mean và Std, do đó **không bị méo bởi ngoại biệt** và **không biến đổi sai lệch các giá trị bằng 0 hay số dư âm**.- Do đó, `RobustScaler` được chọn làm phương pháp chuẩn hóa chính cho nhóm Numerical.

In [29]:
preprocessor, num_cols, cat_cols = build_preprocessing_pipeline(scaler_type='robust')
print("Đã khởi tạo ColumnTransformer với RobustScaler cho Numerical và Passthrough cho Categorical/Ordinal.")

Đã khởi tạo ColumnTransformer với RobustScaler cho Numerical và Passthrough cho Categorical/Ordinal.


---## 12. Fit Preprocessing Pipeline CHỈ TRÊN TẬP TRAIN (Strict Fitting)> [!IMPORTANT]> Thực thi `preprocessor.fit(X_train)`. Tuyệt đối không cho `X_test` tham gia vào bất kỳ thao tác `.fit()` nào.

In [30]:
preprocessor.fit(X_train)
print("Đã FIT Preprocessing Pipeline CHỈ TRÊN tập X_train thành công!")

Đã FIT Preprocessing Pipeline CHỈ TRÊN tập X_train thành công!


---## 13. Transform Tập TRAIN và Tập TESTSử dụng transformer đã fit từ tập TRAIN để transform cho cả `X_train` và `X_test`.

In [31]:
X_train_proc, X_test_proc, fitted_prep = fit_transform_data(preprocessor, X_train, X_test)

print("Kích thước ma trận X_train sau transform:", X_train_proc.shape)
print("Kích thước ma trận X_test sau transform :", X_test_proc.shape)
print("\n5 dòng đầu tiên của X_train đã xử lý:")
display(X_train_proc.head())

Kích thước ma trận X_train sau transform: (23972, 23)
Kích thước ma trận X_test sau transform : (5993, 23)

5 dòng đầu tiên của X_train đã xử lý:


,num__LIMIT_BAL,num__AGE,num__BILL_AMT1,num__BILL_AMT2,num__BILL_AMT3,num__BILL_AMT4,num__BILL_AMT5,num__BILL_AMT6,num__PAY_AMT1,num__PAY_AMT2,num__PAY_AMT3,num__PAY_AMT4,num__PAY_AMT5,num__PAY_AMT6,cat__SEX,cat__EDUCATION,cat__MARRIAGE,cat__PAY_0,cat__PAY_2,cat__PAY_3,cat__PAY_4,cat__PAY_5,cat__PAY_6
27394,0.84,0.08,-0.35,-0.35,-0.35,-0.37,-0.37,-0.36,-0.52,-0.48,-0.44,-0.40,-0.40,-0.39,1.00,2.00,1.00,-2.00,-2.00,-2.00,-2.00,-2.00,-2.00
15341,-0.47,0.23,0.39,0.45,0.50,0.55,0.18,0.19,0.05,-0.12,-0.44,0.51,-0.40,-0.08,2.00,3.00,3.00,2.00,2.00,2.00,2.00,2.00,2.00
26617,-0.32,-0.85,-0.35,-0.34,-0.34,-0.35,-0.36,-0.35,-0.41,-0.38,-0.23,-0.28,-0.40,-0.12,1.00,1.00,2.00,-1.00,-1.00,-1.00,-1.00,0.00,0.00
21600,-0.47,-0.62,-0.35,-0.32,-0.35,-0.36,-0.37,-0.34,-0.01,-0.39,-0.35,-0.40,-0.19,-0.39,2.00,2.00,2.00,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00
11563,0.32,-0.31,0.40,0.46,0.08,1.99,2.34,2.33,4.54,5.52,30.04,4.36,6.61,0.74,1.00,2.00,2.00,-2.00,-2.00,-2.00,-2.00,-2.00,-2.00


---## 14. Kiểm tra Shape và Tính chất Ma trận sau Preprocessing- Số lượng thuộc tính sau transform: **23 thuộc tính** (đúng bằng số lượng thuộc tính đầu vào ban đầu).- Không xuất hiện giá trị NaN hay Inf nào sau phép chuẩn hóa.

In [32]:
print(f"X_train_proc Null Count: {X_train_proc.isnull().sum().sum()}")
print(f"X_test_proc Null Count : {X_test_proc.isnull().sum().sum()}")

X_train_proc Null Count: 0
X_test_proc Null Count : 0


---## 15. Kiểm tra Checklist Chống Rò rỉ Dữ liệu (Data Leakage Verification Checklist)| Tiêu chí Kiểm tra | Trạng thái | Giải thích chi tiết || :--- | :---: | :--- || **1. ID không làm feature** | `[x] PASSED` | Cột `ID` đã bị loại bỏ khỏi không gian thuộc tính $X$. || **2. Split trước khi Fit Scaler** | `[x] PASSED` | `train_test_split` được gọi ở Mục 8, `fit()` scaler ở Mục 12. || **3. Encoder không học từ TEST** | `[x] PASSED` | Không dùng Encoder học tham số từ tập TEST. || **4. Transformation không học từ TEST** | `[x] PASSED` | Scaler chỉ học Median & IQR từ `X_train`. || **5. SMOTE không áp dụng trên TEST** | `[x] PASSED` | Chưa thực hiện SMOTE ở bước này; nếu áp dụng ở bước ML sau sẽ chỉ áp dụng trên TRAIN. || **6. Không dùng Target TEST để Preprocess**| `[x] PASSED` | Biến mục tiêu `y_test` không tham gia vào bất kỳ hàm tính toán nào. || **7. Không dùng Thống kê TEST cho TRAIN** | `[x] PASSED` | Tập TEST hoàn toàn độc lập. |

In [33]:
print("CHECKLIST CHỐNG DATA LEAKAGE: 7/7 TIÊU CHÍ ĐẠT [x] PASSED.")

CHECKLIST CHỐNG DATA LEAKAGE: 7/7 TIÊU CHÍ ĐẠT [x] PASSED.


---## 16. Phân tích Mất cân bằng Lớp & Chuẩn bị Phương án Xử lý (Class Imbalance Strategy)Tỷ lệ mất cân bằng tập TRAIN:- Class 0 (Không vỡ nợ): 18,668 mẫu (77.87%)- Class 1 (Vỡ nợ): 5,304 mẫu (22.13%)- Imbalance Ratio: **3.52 : 1**### 3 Phương án Chiến lược sẽ được đánh giá ở Tuần 6 (Model Training):1. **Phương án 1 (Baseline Unbalanced)**: Huấn luyện mô hình gốc không điều chỉnh trọng số để làm mốc so sánh.2. **Phương án 2 (`class_weight='balanced'`)**: Gán trọng số phạt cao hơn cho sai sót ở Lớp thiểu số (Class 1) trong hàm mất mát của Logistic Regression, Random Forest, XGBoost.3. **Phương án 3 (SMOTE Oversampling trên TRAIN)**: Sinh mẫu giả lập cho Lớp 1 **CHỈ TRÊN TẬP TRAIN**. Tập TEST luôn được giữ nguyên 100% để phản ánh đúng thực tế kinh doanh.

In [34]:
print("Chiến lược Class Imbalance đã sẵn sàng. Không áp dụng bất kỳ resampling nào lên tập TEST.")

Chiến lược Class Imbalance đã sẵn sàng. Không áp dụng bất kỳ resampling nào lên tập TEST.


---## 17. Mã minh họa Pipeline Cân bằng Lớp (Code mẫu cho Tuần ML sau)> [!NOTE]> Đoạn code dưới đây chỉ mang tính chất minh họa cấu trúc `imblearn.pipeline.Pipeline` cho các bước huấn luyện mô hình ở Tuần 6. Ở giai đoạn hiện tại, **KHÔNG huấn luyện hay tối ưu bất kỳ mô hình nào**.

In [35]:
# Cấu trúc Pipeline mẫu cho các tuần sau:
# from imblearn.pipeline import Pipeline as ImbPipeline
# from imblearn.over_sampling import SMOTE
# from sklearn.linear_model import LogisticRegression

# pipeline_demo = ImbPipeline([
#     ('preprocessor', preprocessor),
#     ('smote', SMOTE(random_state=42)),
#     ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
# ])

print("Cấu trúc Pipeline mẫu chuẩn bị cho giai đoạn huấn luyện ML ở Tuần 6.")

Cấu trúc Pipeline mẫu chuẩn bị cho giai đoạn huấn luyện ML ở Tuần 6.


---## 18. Tổng kết & Xuất Dữ liệu Processed (Summary & Data Export)Tập dữ liệu đã qua xử lý được lưu vào thư mục `data/processed/`:- `data/processed/X_train.csv` (23,972 mẫu x 23 thuộc tính)- `data/processed/X_test.csv` (5,993 mẫu x 23 thuộc tính)- `data/processed/y_train.csv` (23,972 nhãn)- `data/processed/y_test.csv` (5,993 nhãn)### Các nội dung TUYỆT ĐỐI CHƯA THỰC HIỆN (Theo đúng Timeline):- [ ] **Feature Engineering (Tuần 5)**: Chưa tạo `UTILIZATION_RATE`, `MEAN_BILL_AMT`, `MEAN_PAY_AMT`, `PAY_RATIO`, `MAX_PAY_DELAY`.- [ ] **Machine Learning Training (Tuần 6)**: Chưa train Logistic Regression, Decision Tree, Random Forest, XGBoost.- [ ] **Hyperparameter Tuning & Selection (Tuần 7)**: Chưa tối ưu siêu tham số.- [ ] **SHAP & Model Interpretability (Tuần 8)**: Chưa giải thích SHAP.- [ ] **Streamlit Deployment (Tuần 9)**: Chưa xây dựng Web App.

In [36]:
save_processed_data(X_train_proc, X_test_proc, y_train, y_test, output_dir=os.path.join('..', 'data', 'processed'))
print("ĐÃ HOÀN THÀNH TOÀN BỘ GIAI ĐOẠN PREPROCESSING TUẦN 4!")

Saved processed datasets to ..\data\processed
ĐÃ HOÀN THÀNH TOÀN BỘ GIAI ĐOẠN PREPROCESSING TUẦN 4!
